<a href="https://colab.research.google.com/github/SorenGrubb/microns-neuroglancer-tool/blob/main/MICrONS_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MICrONS mesh downloader

Root ID &rarr; GLB, straight from the public `seg_m1300` mesh shards. No CAVE token needed:
the mesh bucket is public.

**Corrections made 28 July 2026 (all verified against shard `08cc`):**

1. **Scale was wrong.** The earlier version used `VOXEL_SIZE = [8, 8, 40]`. The mesh `info` file
   says the transform is `[16, 16, 40]` &mdash; these meshes are built from the 16&times;16&times;40 nm mip,
   not the 8&times;8&times;40 base. The old value squashed every mesh to half width in x and y.
   Verified: the nucleus centroid of root `864691135234029401` from `nucleus_detection_v0`
   (924.0, 406.8, 728.0 &micro;m) only falls inside the mesh bounding box under `[16, 16, 40]`.
2. **Missing `2**lod` factor.** The old `reconstruct_lod0` omitted it, so it was correct only at
   LOD 0. All four LODs now reconstruct to the same bounding box.
3. **Nothing is hardcoded.** Sharding bits, quantization and the transform are read from
   `mesh/info` at runtime.
4. **One HTTP request per LOD instead of one per fragment.** A LOD's fragments are contiguous in
   the shard, so LOD 0 is 1 request rather than 360.
5. **No more cmake.** `DracoPy` replaces building `draco_decoder` from source, which saves several
   minutes of Colab startup.

In [ ]:
!pip -q install DracoPy trimesh

import requests, gzip, struct, numpy as np, DracoPy, trimesh

BASE = ("https://storage.googleapis.com/iarpa_microns/"
        "minnie/minnie65/seg_m1300/mesh/")

CFG = {}

def load_mesh_info():
    """Read the authoritative mesh info file. Never hardcode these values."""

    info = requests.get(BASE + "info").json()

    assert info["@type"] == "neuroglancer_multilod_draco", info["@type"]

    sh = info["sharding"]

    assert sh["hash"] == "murmurhash3_x86_128", sh["hash"]

    CFG.update(
        preshift_bits  = sh["preshift_bits"],
        minishard_bits = sh["minishard_bits"],
        shard_bits     = sh["shard_bits"],
        quant          = (1 << info["vertex_quantization_bits"]) - 1,
        # 4x3 row-major affine: "stored model" space -> nanometres
        transform      = np.array(info["transform"], dtype=np.float64).reshape(3, 4),
    )

    print("sharding bits :", CFG["preshift_bits"], CFG["minishard_bits"], CFG["shard_bits"])
    print("quantization  :", info["vertex_quantization_bits"], "bits")
    print("scale (nm)    :", CFG["transform"][0, 0], CFG["transform"][1, 1], CFG["transform"][2, 2])

    return CFG

load_mesh_info()

In [ ]:
def rotl32(x, r):
    x &= 0xffffffff
    return ((x << r) | (x >> (32 - r))) & 0xffffffff


def murmur_mix(h):
    h ^= (h >> 16)
    h = (h * 0x85ebca6b) & 0xffffffff
    h ^= (h >> 13)
    h = (h * 0xc2b2ae35) & 0xffffffff
    h ^= (h >> 16)
    return h & 0xffffffff


def murmur64(input_val):
    """Low 64 bits of murmurhash3_x86_128 over an 8-byte little-endian key."""

    h1 = h2 = h3 = h4 = 0

    c1, c2, c3 = 0x239b961b, 0xab0e9789, 0x38b34ae5

    high32 = (input_val >> 32) & 0xffffffff
    low32  = input_val & 0xffffffff

    k2 = (high32 * c2) & 0xffffffff
    k2 = rotl32(k2, 16)
    k2 = (k2 * c3) & 0xffffffff
    h2 ^= k2

    k1 = (low32 * c1) & 0xffffffff
    k1 = rotl32(k1, 15)
    k1 = (k1 * c2) & 0xffffffff
    h1 ^= k1

    h1 ^= 8; h2 ^= 8; h3 ^= 8; h4 ^= 8   # key length in bytes

    h1 = (h1 + h2 + h3 + h4) & 0xffffffff
    h2 = (h2 + h1) & 0xffffffff
    h3 = (h3 + h1) & 0xffffffff
    h4 = (h4 + h1) & 0xffffffff

    h1 = murmur_mix(h1); h2 = murmur_mix(h2)
    h3 = murmur_mix(h3); h4 = murmur_mix(h4)

    h1 = (h1 + h2 + h3 + h4) & 0xffffffff
    h2 = (h2 + h1) & 0xffffffff

    return h1 | (h2 << 32)


def get_shard_and_minishard(root_id):

    hashed = murmur64(root_id >> CFG["preshift_bits"])

    mask = (1 << (CFG["minishard_bits"] + CFG["shard_bits"])) - 1

    sm = hashed & mask

    minishard = sm & ((1 << CFG["minishard_bits"]) - 1)

    shard = (sm >> CFG["minishard_bits"]) & ((1 << CFG["shard_bits"]) - 1)

    return format(shard, "04x"), minishard


# sanity check against the manually traced neuron
assert get_shard_and_minishard(864691135234029401) == ("08cc", 32)
print("hash check OK")

In [ ]:
def get_range(url, start, end):
    """Inclusive byte range. Refuses a 200, which would mean the whole ~500 MB shard."""

    r = requests.get(url, headers={"Range": f"bytes={start}-{end}"})

    if r.status_code != 206:
        raise RuntimeError(f"expected HTTP 206, got {r.status_code}")

    return r.content


def find_manifest(root_id):

    shard, minishard = get_shard_and_minishard(root_id)

    url = BASE + shard + ".shard"

    shard_index_size = (1 << CFG["minishard_bits"]) * 16

    off = minishard * 16

    entry = get_range(url, off, off + 15)

    start_offset = struct.unpack("<Q", entry[:8])[0]
    end_offset   = struct.unpack("<Q", entry[8:])[0]

    if start_offset == end_offset:
        raise ValueError(f"minishard {minishard} is empty")

    ms_start = shard_index_size + start_offset
    ms_end   = shard_index_size + end_offset

    decoded = gzip.decompress(get_range(url, ms_start, ms_end - 1))

    arr = np.frombuffer(decoded, dtype="<u8")

    n = len(arr) // 3

    ids    = arr[:n].copy()
    starts = arr[n:2 * n].copy()
    sizes  = arr[2 * n:].copy()

    for i in range(1, n):
        ids[i] += ids[i - 1]

    prev = shard_index_size

    for i in range(n):
        starts[i] += prev
        prev = starts[i] + sizes[i]

    matches = np.where(ids == root_id)[0]

    if len(matches) == 0:
        raise ValueError(f"root ID {root_id} not found in minishard {minishard}")

    idx = int(matches[0])

    return {
        "shard": shard,
        "minishard": minishard,
        "manifest_start": int(starts[idx]),
        "manifest_size": int(sizes[idx]),
        "url": url,
    }

In [ ]:
def parse_manifest(manifest):

    ptr = 0

    chunk_shape = struct.unpack_from("<3f", manifest, ptr); ptr += 12
    grid_origin = struct.unpack_from("<3f", manifest, ptr); ptr += 12

    num_lods = struct.unpack_from("<I", manifest, ptr)[0]; ptr += 4

    lod_scales = struct.unpack_from(f"<{num_lods}f", manifest, ptr)
    ptr += 4 * num_lods

    vertex_offsets = np.frombuffer(
        manifest, dtype="<f4", count=num_lods * 3, offset=ptr
    ).reshape(num_lods, 3)
    ptr += num_lods * 12

    num_fragments_per_lod = struct.unpack_from(f"<{num_lods}I", manifest, ptr)
    ptr += num_lods * 4

    lods = []

    for lod in range(num_lods):

        n = num_fragments_per_lod[lod]

        positions = np.frombuffer(
            manifest, dtype="<u4", count=n * 3, offset=ptr
        ).reshape(3, n)
        ptr += n * 12

        sizes = np.frombuffer(manifest, dtype="<u4", count=n, offset=ptr)
        ptr += n * 4

        lods.append({"positions": positions, "sizes": sizes})

    return {
        "chunk_shape": chunk_shape,
        "grid_origin": grid_origin,
        "lod_scales": lod_scales,
        "vertex_offsets": vertex_offsets,
        "num_fragments_per_lod": num_fragments_per_lod,
        "lods": lods,
    }


def fragment_layout(parsed, manifest_start):
    """Byte range of every fragment. Fragments precede the manifest, LOD 0 first."""

    total = sum(int(s) for lod in parsed["lods"] for s in lod["sizes"])

    offset = manifest_start - total

    per_lod = []

    for lod in parsed["lods"]:

        start = offset
        frags = []

        for s in lod["sizes"].tolist():
            frags.append((offset, s))
            offset += s

        per_lod.append({"start": start, "end": offset - 1,
                        "bytes": offset - start, "frags": frags})

    return per_lod

In [ ]:
def reconstruct(parsed, layout, blob, lod, units="um"):
    """Decode one LOD into a single trimesh, in absolute dataset coordinates."""

    cs = np.array(parsed["chunk_shape"], dtype=np.float64)
    go = np.array(parsed["grid_origin"], dtype=np.float64)
    vo = parsed["vertex_offsets"][lod].astype(np.float64)

    positions = parsed["lods"][lod]["positions"]

    L = layout[lod]

    quant = CFG["quant"]

    # >>> the factor that was missing before: chunk size doubles with every LOD
    lod_scale = 2 ** lod

    T = CFG["transform"]            # 3x4 affine, stored model space -> nm

    divisor = 1000.0 if units == "um" else 1.0

    meshes = []

    for i, (off, size) in enumerate(L["frags"]):

        if size == 0:
            continue

        frag = blob[off - L["start"] : off - L["start"] + size]

        m = DracoPy.decode(frag)

        v = np.asarray(m.points, dtype=np.float64)

        fpos = positions[:, i].astype(np.float64)

        # neuroglancer multi-resolution mesh spec:
        #   grid_origin + vertex_offsets[lod] + chunk_shape * 2**lod * (fragPos + x/quant)
        v = go + vo + cs * lod_scale * (fpos + v / quant)

        # then the info file's affine, into nanometres
        v = v @ T[:, :3].T + T[:, 3]

        meshes.append(trimesh.Trimesh(vertices=v / divisor,
                                      faces=np.asarray(m.faces),
                                      process=False))

    combined = trimesh.util.concatenate(meshes)

    size = combined.bounds[1] - combined.bounds[0]

    print(f"LOD {lod}: {len(combined.vertices):,} vertices, "
          f"{len(combined.faces):,} faces")
    print(f"bounding box ({units}): "
          f"{size[0]:.1f} x {size[1]:.1f} x {size[2]:.1f}")

    return combined

In [ ]:
def extract_neuron(root_id, lod=0, units="um", outfile=None):
    """Root ID -> GLB. lod 0 is full resolution; higher is coarser and much lighter."""

    if not CFG:
        load_mesh_info()

    info = find_manifest(root_id)

    print(f"shard {info['shard']}.shard, minishard {info['minishard']}")

    manifest = gzip.decompress(
        get_range(info["url"], info["manifest_start"],
                  info["manifest_start"] + info["manifest_size"] - 1)
    )

    parsed = parse_manifest(manifest)

    layout = fragment_layout(parsed, info["manifest_start"])

    print("\nfragments per LOD:")

    for l, L in enumerate(layout):
        print(f"  LOD {l}: {len(L['frags']):>4} fragments, "
              f"{L['bytes'] / 1048576:6.2f} MB")

    L = layout[lod]

    print(f"\nfetching LOD {lod} in one request "
          f"({L['bytes'] / 1048576:.2f} MB)...")

    blob = get_range(info["url"], L["start"], L["end"])

    mesh = reconstruct(parsed, layout, blob, lod, units=units)

    if outfile is None:
        outfile = f"/content/microns_{root_id}_lod{lod}_{units}.glb"

    mesh.export(outfile)

    print("\nsaved:", outfile)

    return outfile, mesh

In [ ]:
ROOT_ID = 864691135234029401

glb, mesh = extract_neuron(ROOT_ID, lod=2)

### Check: every LOD must give the same bounding box

If the `2**lod` factor or the transform were wrong, the coarse LODs would not line up with LOD 0.

In [ ]:
info = find_manifest(ROOT_ID)

manifest = gzip.decompress(
    get_range(info["url"], info["manifest_start"],
              info["manifest_start"] + info["manifest_size"] - 1)
)

parsed = parse_manifest(manifest)
layout = fragment_layout(parsed, info["manifest_start"])

for lod in range(len(layout)):

    L = layout[lod]

    blob = get_range(info["url"], L["start"], L["end"])

    reconstruct(parsed, layout, blob, lod)

    print()